In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / "data" / "raw"
print("Project root:", ROOT)

for name in ["rainfall_2022_2026.csv", "rainfall_data.csv", "lucknow_grid_elevation.csv"]:
    df = pd.read_csv(RAW / name)
    print("\n=====", name, df.shape)
    print(df.dtypes)
    print(df.head(3))
    print(df.tail(2))

labels = pd.read_csv(ROOT / "data" / "labels" / "waterlogging_reports.csv")
print("\n===== labels", labels.shape)
print(labels)

PROC = ROOT / "data" / "processed"

r = pd.read_csv(RAW / "rainfall_2022_2026.csv", parse_dates=["time"]).set_index("time")
r = r.asfreq("h")                                   # force a strict hourly index
print("missing hours:", r["precipitation"].isna().sum())

rain = pd.DataFrame(index=r.index)
rain["rain_1h"] = r["precipitation"]
for h in [3, 6, 24, 72]:
    rain[f"rain_{h}h"] = r["precipitation"].rolling(h, min_periods=h).sum()

rain.to_csv(PROC / "rain_features.csv")
print(rain.shape)
print(rain.loc["2026-09-24":"2026-09-26"].describe().round(1))
print("Hourly rain peak on 25 Sep:", rain.loc["2026-09-25", "rain_1h"].idxmax())

print("missing hours:", r["precipitation"].isna().sum())
print(rain.shape)
print(rain.loc["2026-09-25", ["rain_1h","rain_24h"]].round(1).iloc[[0, 8, 14, 20, 23]])



Project root: c:\Users\Ausaaf Siddiqui\AquaRoute-AI

===== rainfall_2022_2026.csv (41880, 4)
time                        str
precipitation           float64
temperature_2m          float64
relative_humidity_2m      int64
dtype: object
                  time  precipitation  temperature_2m  relative_humidity_2m
0  2022-01-01 00:00:00            0.0            11.1                    84
1  2022-01-01 01:00:00            0.0            10.2                    88
2  2022-01-01 02:00:00            0.0            10.7                    84
                      time  precipitation  temperature_2m  \
41878  2026-10-11 22:00:00            0.0            25.0   
41879  2026-10-11 23:00:00            0.0            24.9   

       relative_humidity_2m  
41878                    83  
41879                    87  

===== rainfall_data.csv (6816, 4)
time                        str
precipitation           float64
temperature_2m          float64
relative_humidity_2m      int64
dtype: object
          

In [ ]:
print("missing hours:", r["precipitation"].isna().sum(), "| shape:", rain.shape)

e = pd.read_csv(RAW / "lucknow_grid_elevation.csv")
print(e.shape); print(e.dtypes); print(e.head(3))

labels = pd.read_csv(ROOT / "data" / "labels" / "waterlogging_reports.csv")
print(labels)

missing hours: 0 | shape: (41880, 5)
(483, 3)
lat          float64
lon          float64
elevation    float64
dtype: object
     lat    lon  elevation
0  26.76  80.85      128.0
1  26.76  80.86      125.0
2  26.76  80.87      122.0
         date location_name      lat      lon severity    source  \
0  2026-09-25    Hazratganj  26.8548  80.9448     high  NewsDrum   
1  2026-09-25   Gomti Nagar  26.8612  81.0039     high  NewsDrum   
2  2026-09-25       Aliganj  26.9051  80.9480     high  NewsDrum   
3  2026-09-25      Charbagh  26.8335  80.9264     high  NewsDrum   
4  2026-09-25         Chowk  26.8677  80.9042     high  NewsDrum   
5  2026-09-25  Indira Nagar  26.8823  80.9900     high  NewsDrum   
6  2026-09-25      Alambagh  26.8140  80.9025     high  NewsDrum   

                                          source_url  
0  https://www.newsdrum.in/national/late-septembe...  
1  https://www.newsdrum.in/national/late-septembe...  
2  https://www.newsdrum.in/national/late-septembe...  
3  h

In [ ]:
ROOT=Path.cwd()
while not (ROOT/".git").exists() and ROOT!=ROOT.parent:
    ROOT=ROOT.parent
RAW,PROC=ROOT/"data"/"raw",ROOT/"data"/"processed"

e=pd.read_csv(RAW/"lucknow_grid_elevation.csv")
coords, elev = e[["lat", "lon"]].to_numpy(), e["elevation"].to_numpy()
dist = np.sqrt(((coords[:, None, :] - coords[None, :, :]) ** 2).sum(-1))

mask = (dist > 0) & (dist <= 0.025)          # neighbours within ~2.5 km, self excluded
e["relative_elev"] = elev - (mask * elev[None, :]).sum(1) / mask.sum(1)

e.to_csv(PROC / "grid_terrain.csv", index=False)
print(e["relative_elev"].describe().round(2))

count    483.00
mean       0.04
std        2.82
min      -12.15
25%       -1.57
50%       -0.25
75%        1.70
max       12.93
Name: relative_elev, dtype: float64


In [ ]:


ROOT = Path.cwd()
while not (ROOT / ".git").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
PROC = ROOT / "data" / "processed"

rain = pd.read_csv(PROC / "rain_features.csv", index_col=0, parse_dates=True)
grid = pd.read_csv(PROC / "grid_terrain.csv")
labels = pd.read_csv(ROOT / "data" / "labels" / "waterlogging_reports.csv", parse_dates=["date"])

labels["label"] = (labels["severity"].str.lower() != "none").astype(int)

rows = []
for _, r in labels.iterrows():
    day = rain[rain.index.strftime("%Y-%m-%d") == r["date"].strftime("%Y-%m-%d")]
    if day.empty:
        print("No rain data for", r["date"].date()); continue
    hour = day["rain_1h"].idxmax()           # the fix

    d = (grid["lat"] - r["lat"]) ** 2 + (grid["lon"] - r["lon"]) ** 2
    if np.sqrt(d.min()) > 0.01:
        print("Skipping (too far from grid):", r["location_name"]); continue
    t = grid.loc[d.idxmin()]

    rows.append({
        "location_name": r["location_name"], "event_hour": hour, "label": r["label"],
        **rain.loc[hour, ["rain_1h", "rain_3h", "rain_6h", "rain_24h", "rain_72h"]].to_dict(),
        "elevation": t["elevation"], "relative_elev": t["relative_elev"],
    })

table = pd.DataFrame(rows)
table.to_csv(PROC / "training_table.csv", index=False)
print(table.round({"rain_1h": 1, "rain_3h": 1, "rain_6h": 1, "rain_24h": 1, "rain_72h": 1,
                   "elevation": 0, "relative_elev": 1}))
print("\nLabel counts:\n", table["label"].value_counts())

  location_name          event_hour  label  rain_1h  rain_3h  rain_6h  \
0    Hazratganj 2026-09-25 20:00:00      1     13.0     33.2     50.1   
1   Gomti Nagar 2026-09-25 20:00:00      1     13.0     33.2     50.1   
2       Aliganj 2026-09-25 20:00:00      1     13.0     33.2     50.1   
3      Charbagh 2026-09-25 20:00:00      1     13.0     33.2     50.1   
4         Chowk 2026-09-25 20:00:00      1     13.0     33.2     50.1   
5  Indira Nagar 2026-09-25 20:00:00      1     13.0     33.2     50.1   
6      Alambagh 2026-09-25 20:00:00      1     13.0     33.2     50.1   

   rain_24h  rain_72h  elevation  relative_elev  
0      76.4      93.6      109.0           -8.8  
1      76.4      93.6      122.0            5.8  
2      76.4      93.6      120.0            0.9  
3      76.4      93.6      123.0            0.1  
4      76.4      93.6      118.0           -1.0  
5      76.4      93.6      122.0            2.4  
6      76.4      93.6      126.0            1.2  

Label counts:


In [ ]:
rain = pd.read_csv(PROC / "rain_features.csv", index_col=0, parse_dates=True)
daily = rain["rain_1h"].resample("D").sum()
peak_hr = rain["rain_1h"].groupby(rain.index.date).idxmax()

top = daily.sort_values(ascending=False).head(15).round(1).to_frame("day_total_mm")
top["peak_hour"] = [peak_hr[d.date()].strftime("%H:%M") for d in top.index]
print(top.sort_index())

            day_total_mm peak_hour
time                              
2022-06-30          47.7     05:00
2022-09-14          40.2     03:00
2022-09-15          82.5     21:00
2023-07-01          39.4     18:00
2024-07-01          58.6     23:00
2024-07-02          80.0     07:00
2024-07-05          44.2     17:00
2024-08-09          40.7     13:00
2024-09-11          57.7     15:00
2024-09-12          57.9     20:00
2025-06-30          38.7     14:00
2025-08-03          40.8     07:00
2025-08-04          66.8     03:00
2026-09-25          93.8     20:00
2026-09-26          41.1     03:00


In [ ]:
rain = pd.read_csv(PROC / "rain_features.csv", index_col=0, parse_dates=True)
d = rain.loc["2024-07-05":"2024-07-08"]
print(d["rain_1h"].resample("D").sum().round(1))
print(d.loc["2024-07-06 18:00":"2024-07-07 12:00", ["rain_1h", "rain_6h", "rain_24h", "rain_72h"]].round(1))

time
2024-07-05    44.2
2024-07-06    33.2
2024-07-07    12.2
2024-07-08     0.6
Freq: D, Name: rain_1h, dtype: float64
                     rain_1h  rain_6h  rain_24h  rain_72h
time                                                     
2024-07-06 18:00:00      0.6     13.2      31.1      82.2
2024-07-06 19:00:00      0.7     13.1      31.0      82.7
2024-07-06 20:00:00      0.1     13.1      31.0      82.6
2024-07-06 21:00:00      0.7     12.2      31.6      83.3
2024-07-06 22:00:00      2.9      9.3      33.7      86.2
2024-07-06 23:00:00      1.8      6.8      33.2      87.8
2024-07-07 00:00:00      0.8      7.0      31.1      87.8
2024-07-07 01:00:00      0.1      6.4      30.8      87.6
2024-07-07 02:00:00      0.1      6.4      30.6      87.2
2024-07-07 03:00:00      0.0      5.7      30.3      86.4
2024-07-07 04:00:00      0.0      2.8      30.2      82.5
2024-07-07 05:00:00      0.1      1.1      30.2      81.5
2024-07-07 06:00:00      2.5      2.8      32.2      83.6
2024-07-07

In [ ]:
g = pd.read_csv(PROC / "grid_terrain.csv")
print("lat", g["lat"].min(), "-", g["lat"].max(), "| lon", g["lon"].min(), "-", g["lon"].max())

lat 26.76 - 26.95800000000001 | lon 80.85 - 81.0500000000001


In [ ]:
g = pd.read_csv(PROC / "grid_terrain.csv")

def check(name, lat, lon):
    dist = np.sqrt(((g["lat"] - lat)**2 + (g["lon"] - lon)**2).min())
    print(f"{name:<15} nearest grid point is {dist:.4f} deg away ->", "OK" if dist <= 0.01 else "OUTSIDE GRID")

check("Ashiyana",       26.7996, 80.9174)
check("Faizullaganj",   26.8991, 80.9286)
check("Sarojini Nagar", 26.7475, 80.8639)

Aashiana        nearest grid point is 1.1402 deg away -> OUTSIDE GRID
Faizullaganj    nearest grid point is 1.1402 deg away -> OUTSIDE GRID
Sarojini Nagar  nearest grid point is 1.1402 deg away -> OUTSIDE GRID
